In [15]:
from __future__ import annotations
import importlib
import numpy as np
import pandas as pd

In [21]:
def print_matrix(M):
    print(pd.DataFrame(D2).to_string(index=False, header=False, float_format=lambda x: f"{x:.4f}"))

In [2]:
def squared_l2_distance_from_gram(gram: np.ndarray) -> np.ndarray:
    diagonal = np.diag(gram)
    return np.maximum(diagonal[:, None] + diagonal[None, :] - 2.0 * gram, 0.0)

In [3]:
def exact_gaussian_kde_l2_gram(
        samples: np.ndarray,
        h_grid: np.ndarray,
) -> np.ndarray:
    samples = np.asarray(samples, dtype=float)
    h_grid = np.asarray(h_grid, dtype=float)
    n = len(samples)
    squared_differences = (samples[:, None] - samples[None, :]) ** 2
    gram = np.empty((len(h_grid), len(h_grid)), dtype=float)
    N = len(h_grid)
    for i in range(N):
        for j in range(i, N):
            h_i = h_grid[i]
            h_j = h_grid[j]
            variance = h_i**2 + h_j**2
            value = np.exp(-squared_differences / (2.0 * variance)).sum()
            value /= n**2 * np.sqrt(2.0 * np.pi * variance)
            gram[i, j] = value
            gram[j, i] = value
    return gram

In [4]:
def beta2_divergence_matrix(
        samples: np.ndarray,
        h_grid: np.ndarray,
) -> np.ndarray:
    gram = exact_gaussian_kde_l2_gram(samples=samples, h_grid=h_grid)
    return 0.5 * squared_l2_distance_from_gram(gram=gram)

In [24]:
def beta2_similarity_matrix(
        samples: np.ndarray,
        h_grid: np.ndarray,
        tau: float = 1.0,
) -> np.ndarray:
    if tau <= 0.0:
        raise ValueError("tau must be positive.")
    divergence = beta2_divergence_matrix(samples=samples, h_grid=h_grid)
    similarity = np.exp(-float(tau) * divergence)
    np.fill_diagonal(a=similarity, val=1.0)
    return similarity

In [30]:
rng = np.random.default_rng(12345)
n = 200
N = 6
samples = rng.normal(loc=0.0, scale=1.0, size=n)
h_grid = np.geomspace(start=0.1, stop=1.0, num=N)
print(h_grid)
D2 = beta2_divergence_matrix(samples=samples, h_grid=h_grid)
SimMatr = beta2_similarity_matrix(samples=samples, h_grid=h_grid, tau=100.0)
print()
print_matrix(D2)
print()
print_matrix(SimMatr)

[0.1        0.15848932 0.25118864 0.39810717 0.63095734 1.        ]

0.0000 0.0014 0.0040 0.0066 0.0103 0.0193
0.0014 0.0000 0.0008 0.0027 0.0061 0.0148
0.0040 0.0008 0.0000 0.0007 0.0033 0.0113
0.0066 0.0027 0.0007 0.0000 0.0012 0.0075
0.0103 0.0061 0.0033 0.0012 0.0000 0.0029
0.0193 0.0148 0.0113 0.0075 0.0029 0.0000

0.0000 0.0014 0.0040 0.0066 0.0103 0.0193
0.0014 0.0000 0.0008 0.0027 0.0061 0.0148
0.0040 0.0008 0.0000 0.0007 0.0033 0.0113
0.0066 0.0027 0.0007 0.0000 0.0012 0.0075
0.0103 0.0061 0.0033 0.0012 0.0000 0.0029
0.0193 0.0148 0.0113 0.0075 0.0029 0.0000


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from kde_tools.bandwidths import (
    gaussian_mise_oracle_bandwidth,
    nearest_bandwidth_grid_index,
)
from kde_tools.fourier import (
    beta2_divergence_matrix_fft,
    beta2_similarity_matrix_fft,
)
import kde_tools.similarities as kde_similarities

# Reload helpers that may have changed since this kernel was started.
importlib.reload(kde_similarities)
from kde_tools.similarities import (
    beta2_divergence_matrix,
    doob_stationary_mass,
    first_order_perron_vector,
    perron_vector,
    population_gaussian_beta2_divergence_matrix,
)

np.set_printoptions(precision=5, suppress=True)
seed = 20260818
